# Working with H5 Files in v4.0
® *Copyright Bimea 2024-2026*

This notebook demonstrates how to record and replay acquisitions using the MuH5 format in v4.0.

## What is MuH5?

**MuH5** is Megamicros' custom HDF5 format for storing:
- MEMS microphone signals
- Analog input signals  
- Synchronous video frames
- Acquisition metadata (sampling frequency, MEMS positions, etc.)

## v4.0 Improvements

✨ **Unified API**: Use `Megamicros(filepath='...')` to replay H5 files  
✨ **Same interface**: Replay works exactly like live acquisition  
✨ **Read-only mode**: Use legacy `MuH5` class for metadata extraction  
✨ **Video support**: Extract and display synchronized video frames  
✨ **Frame-by-frame**: Efficient iteration over large files  

Let's explore both recording and replaying workflows!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import megamicros

from megamicros import Megamicros
from megamicros.muh5 import MuH5
from megamicros.log import log

log.setLevel("INFO")

print(f"Megamicros version: {megamicros.__version__}")

## Recording to H5 Files

**Note**: This example shows the concept. Full recording implementation requires additional tools.

For now, use the C++ library (`megamicros-libcc`) or existing v3.x tools to create `.muh5` files.
We'll focus on **replaying** existing H5 files with v4.0.

In [ ]:
# Example recording workflow (conceptual - not yet fully implemented in v4.0)

"""
# Record from USB hardware to H5 file
from megamicros import Megamicros

antenna = Megamicros(usb=True)

# Run acquisition
antenna.run(
    mems=antenna.available_mems,
    duration=10,
    sampling_frequency=50000
)

# Save to H5 file (requires additional implementation)
# antenna.save_to_h5('recording.muh5')

antenna.wait()
"""

print("⚠ Full H5 recording from Megamicros class coming in future v4.x release")
print("💡 For now, use megamicros-libcc tools to create .muh5 files")- Then replay them with v4.0 (see below)")

## Replaying H5 Files (NEW in v4.0)

The easiest way to work with H5 files is to use `Megamicros(filepath='...')`:

This automatically selects **H5DataSource** and provides the same API as live acquisition!

In [ ]:
# Path to an existing .muh5 file
# Replace with your own file path
H5_FILE_PATH = "/Users/brunogas/Documents/Dev/Bimea/megamicros-libcc/build/muh5-20251107-123126.h5"

try:
    # Load H5 file as a data source
    antenna_replay = Megamicros(filepath=H5_FILE_PATH)
    
    # Get info
    info = antenna_replay.infos
    print(f"\n✓ H5 file loaded successfully!")
    print(f"  Source type: {info['source_type']}")
    print(f"  Sampling frequency: {antenna_replay.sampling_frequency} Hz")
    print(f"  Available MEMS: {len(antenna_replay.available_mems)}")
    print(f"  Duration: {info.get('duration', 'N/A')}s")
    
    h5_file_exists = True
    
except FileNotFoundError:
    print(f"⚠ H5 file not found: {H5_FILE_PATH}")
    print("💡 Update H5_FILE_PATH to point to your own .muh5 file")
    print("💡 Or use megamicros-libcc to create test files")
    h5_file_exists = False

## Replaying Frames

If the H5 file exists, replay it exactly like a live acquisition:

In [ ]:
if h5_file_exists:
    # Replay the entire file
    antenna_replay.run()
    
    # Collect frames in real-time
    frame_count = 0
    total_samples = 0
    
    for frame in antenna_replay:
        frame_count += 1
        total_samples += frame.shape[1]
        
        if frame_count <= 3:  # Show first 3 frames
            print(f"  Frame {frame_count}: shape={frame.shape}, dtype={frame.dtype}")
    
    antenna_replay.wait()
    
    print(f"\n✓ Replayed {frame_count} frames")
    print(f"✓ Total samples: {total_samples}")
    print(f"✓ Duration: {total_samples / antenna_replay.sampling_frequency:.2f}s")
else:
    print("⚠ Skipping replay (no H5 file)")

## Selective Channel Replay

You can select specific MEMS channels when replaying:

In [ ]:
if h5_file_exists:
    # Replay only specific MEMS
    antenna_replay.run(
        mems=[0, 1, 2, 3],  # Only first 4 MEMS
        duration=1.0        # Only first second
    )
    
    # Collect selected channels
    frames = []
    for frame in antenna_replay:
        frames.append(frame)
    
    antenna_replay.wait()
    
    # Concatenate
    signal = np.concatenate(frames, axis=1)
    
    print(f"✓ Replayed signal: {signal.shape}")
    print(f"  Channels: {signal.shape[0]} (selected MEMS)")
    print(f"  Samples: {signal.shape[1]}")
    
    # Plot first MEMS
    t = np.arange(signal.shape[1]) / antenna_replay.sampling_frequency
    
    plt.figure(figsize=(12, 4))
    plt.plot(t, signal[0, :])
    plt.title('MEMS 0 - Replayed from H5 File')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Skipping selective replay (no H5 file)")

## Read-Only H5 Access (Legacy MuH5 Class)

For advanced metadata extraction and video access, use the legacy `MuH5` class:

**Use this when you only need to read metadata or extract video, not replay signals.**

In [ ]:
if h5_file_exists:
    # Open in read-only mode
    file = MuH5(H5_FILE_PATH)
    
    print("\nH5 File Metadata:")
    print(f"  Sampling frequency: {file.sampling_frequency} Hz")
    print(f"  MEMS count: {file.mems_number}")
    print(f"  Total channels: {file.channels_number}")
    print(f"  Duration: {file.duration:.2f}s")
    print(f"  Total samples: {file.samples_number}")
    print(f"  Available MEMS: {file.available_mems[:8]}... (showing first 8)")
    print(f"\nVideo Information:")
    print(f"  Video available: {file.video_available}")
    
    if file.video_available:
        print(f"  Adaptive FPS: {file.video_adaptive_fps}")
        print(f"  Max FPS: {file.video_max_fps}")
        print(f"  Dataset count: {file.video_dataset_count}")
        print(f"  Frame count: {file.video_frame_count}")
    
    video_available = file.video_available
else:
    print("⚠ Skipping metadata extraction (no H5 file)")
    video_available = False

## Extract and Display Video Frames

If the H5 file contains synchronized video:

In [ ]:
if h5_file_exists and video_available:
    # Extract first frame
    first_frame = file.get_video_frames(start_frame=0, end_frame=0)
    
    if len(first_frame) > 0:
        plt.figure(figsize=(10, 8))
        plt.imshow(first_frame[0])
        plt.title('First Video Frame from H5 File')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        print(f"✓ Video frame shape: {first_frame[0].shape}")
    else:
        print("⚠ No video frames could be extracted")
        
elif h5_file_exists and not video_available:
    print("⚠ This H5 file does not contain video data")
else:
    print("⚠ Skipping video display (no H5 file)")

## Export Video to AVI

Extract all video frames and create a standard video file:

**Note**: Requires `opencv-python` → `pip install opencv-python`

In [ ]:
if h5_file_exists and video_available:
    try:
        import cv2
        
        # Extract all video frames
        print("Extracting video frames...")
        all_frames = file.get_video_frames()
        
        if len(all_frames) > 0:
            # Get frame dimensions
            height, width = all_frames[0].shape[:2]
            fps = 30  # Or use file.video_max_fps
            
            # Create video writer
            output_path = 'extracted_video.avi'
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            video_out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
            
            # Write frames
            for i, frame in enumerate(all_frames):
                video_out.write(frame)
                
                if (i + 1) % 100 == 0:
                    print(f"  Processed {i + 1}/{len(all_frames)} frames...")
            
            video_out.release()
            
            print(f"✓ Video exported to: {output_path}")
            print(f"  Frames: {len(all_frames)}")
            print(f"  Resolution: {width}x{height}")
            print(f"  FPS: {fps}")
            
            # Optional: Display video (comment out if running headless)
            """
            print("\nPress 'q' to stop playback...")
            cap = cv2.VideoCapture(output_path)
            
            while cap.isOpened():
                ret, frame = cap.read()
                if ret:
                    cv2.imshow('Extracted Video', frame)
                    if cv2.waitKey(25) & 0xFF == ord('q'):
                        break
                else:
                    break
            
            cap.release()
            cv2.destroyAllWindows()
            """
        else:
            print("⚠ No video frames found in file")
            
    except ImportError:
        print("⚠ OpenCV not installed")
        print("  Install with: pip install opencv-python")
        
elif h5_file_exists and not video_available:
    print("⚠ This H5 file does not contain video data")
else:
    print("⚠ Skipping video export (no H5 file)")

## Computing Signal Statistics from H5

Process large H5 files efficiently by iterating frame-by-frame:

In [ ]:
if h5_file_exists:
    # Compute statistics without loading entire file
    antenna_stats = Megamicros(filepath=H5_FILE_PATH)
    
    antenna_stats.run(mems=[0])  # Process only MEMS 0
    
    # Compute running statistics
    sample_count = 0
    sum_signal = 0.0
    sum_squared = 0.0
    min_val = np.inf
    max_val = -np.inf
    
    for frame in antenna_stats:
        data = frame[0, :]  # MEMS 0 channel
        
        sample_count += len(data)
        sum_signal += np.sum(data)
        sum_squared += np.sum(data ** 2)
        min_val = min(min_val, np.min(data))
        max_val = max(max_val, np.max(data))
    
    antenna_stats.wait()
    
    # Calculate statistics
    mean = sum_signal / sample_count
    variance = (sum_squared / sample_count) - (mean ** 2)
    std = np.sqrt(variance)
    
    print(f"\nMEMS 0 Statistics (from H5):")
    print(f"  Samples: {sample_count}")
    print(f"  Mean: {mean:.2e}")
    print(f"  Std Dev: {std:.2e}")
    print(f"  Min: {min_val:.2e}")
    print(f"  Max: {max_val:.2e}")
    print(f"  Dynamic range: {max_val - min_val:.2e}")
else:
    print("⚠ Skipping statistics (no H5 file)")

## Creating H5 Files (Future Feature)

**Coming soon in v4.x**: Direct H5 recording from `Megamicros` class

```python
# Future API (not yet implemented)
from megamicros import Megamicros, H5Writer

antenna = Megamicros(usb=True)

# Option 1: Record during acquisition
with H5Writer('recording.muh5') as writer:
    antenna.run(duration=10, mems=antenna.available_mems)
    
    for frame in antenna:
        writer.write_frame(frame)
    
    antenna.wait()

# Option 2: Record with video
with H5Writer('recording_with_video.muh5', video=True) as writer:
    antenna.run(duration=10)
    
    for frame in antenna:
        writer.write_frame(frame)
        writer.add_video_frame(camera.capture())  # From external camera
    
    antenna.wait()
```

**For now**: Use `megamicros-libcc` C++ tools to create `.muh5` files.

## Comparing Live vs Replay

The beauty of v4.0 is that **the same code works for live and replay**:

In [ ]:
def process_acquisition(antenna, description):
    """Process acquisition from any source (USB, H5, Random, WebSocket)"""
    
    antenna.run(mems=[0, 1], duration=1.0)
    
    # Collect energy
    energy_per_frame = []
    
    for frame in antenna:
        energy = np.mean(frame ** 2)
        energy_per_frame.append(energy)
    
    antenna.wait()
    
    avg_energy = np.mean(energy_per_frame)
    
    print(f"\n{description}:")
    print(f"  Frames: {len(energy_per_frame)}")
    print(f"  Avg energy: {avg_energy:.2e}")
    
    return energy_per_frame

# Same function works for all sources!

# Option 1: Live USB
# antenna_live = Megamicros(usb=True)
# process_acquisition(antenna_live, "Live USB")

# Option 2: H5 replay
if h5_file_exists:
    antenna_h5 = Megamicros(filepath=H5_FILE_PATH)
    energy_h5 = process_acquisition(antenna_h5, "H5 Replay")

# Option 3: Random (always works)
antenna_random = Megamicros()  # Falls back to Random if no USB
energy_random = process_acquisition(antenna_random, f"Random Source ({antenna_random.infos['source_type']})")

print("\n✓ Same code, multiple sources!")

## Key Takeaways

✅ **Replay H5 files**: `Megamicros(filepath='file.muh5')`  
✅ **Same API**: Replay uses identical interface to live acquisition  
✅ **Metadata extraction**: Use `MuH5` class for read-only access  
✅ **Video support**: Extract synchronized video frames  
✅ **Efficient processing**: Frame-by-frame iteration, no full load  
✅ **Source-agnostic code**: Write once, run on USB/H5/Random/WebSocket  

**Next Steps**:
- Record acquisitions using `megamicros-libcc` tools
- Replay and analyze with v4.0 Python API
- Combine with beamforming (Notebook 04) for offline analysis

**See `USER_GUIDE_v4.md` for complete H5 documentation!**